In [1]:
!sudo apt-get update -y
!sudo apt-get install python3.9 python3.9-distutils -y
!wget https://bootstrap.pypa.io/get-pip.py
!python3.9 get-pip.py
!python3.9 -m pip install mediapipe==0.10.9 opencv-python-headless pandas patool numpy

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,309 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,803 kB]
Get:13 https://r2u.stat.illinois.edu/ubu

In [4]:
%%writefile isleyici.py
import os
os.environ['MPLBACKEND'] = 'Agg'
import cv2
import mediapipe as mp
import pandas as pd
import patoolib
import sys

# AYARLAR
RAR_PATH = '/content/drive/MyDrive/test.rar'
CSV_PATH = '/content/drive/MyDrive/test.csv'
EXTRACT_FOLDER = '/content/test_videolari'
OUTPUT_FILE = '/content/drive/MyDrive/video_koordinat_veriseti_TEST.csv'

# 1. RAR ÇIKARMA
if not os.path.exists(EXTRACT_FOLDER):
    os.makedirs(EXTRACT_FOLDER)
    print("--- RAR arşivi açılıyor... ---")
    try:
        patoolib.extract_archive(RAR_PATH, outdir=EXTRACT_FOLDER)
    except Exception as e:
        print(f"Hata: {e}")

# 2. ETİKETLERİ YÜKLE
df_labels = pd.read_csv(CSV_PATH, names=['video_name', 'class_id'])
label_dict = dict(zip(df_labels['video_name'], df_labels['class_id']))

# 3. SÜTUNLARI HAZIRLA VE BOŞ DOSYAYI OLUŞTUR (Veya mevcut olanı kontrol et)
cols = ['video_name', 'class_id', 'frame_no']
cols += [f'pose_{i}_{axis}' for i in range(33) for axis in ('x','y','z')]
cols += [f'rh_{i}_{axis}' for i in range(21) for axis in ('x','y','z')]
cols += [f'lh_{i}_{axis}' for i in range(21) for axis in ('x','y','z')]

# ZATEN İŞLENMİŞ VİDEOLARI BUL (Kaldığı yerden devam etme özelliği)
processed_videos = []
if os.path.exists(OUTPUT_FILE):
    try:
        df_check = pd.read_csv(OUTPUT_FILE, usecols=['video_name'])
        processed_videos = df_check['video_name'].unique().tolist()
        print(f"--- {len(processed_videos)} video zaten işlenmiş. Devam ediliyor... ---")
    except:
        pd.DataFrame(columns=cols).to_csv(OUTPUT_FILE, index=False)
else:
    pd.DataFrame(columns=cols).to_csv(OUTPUT_FILE, index=False)

# 4. MEDIAPIPE VE İŞLEME
mp_holistic = mp.solutions.holistic
to_process = [v for v in label_dict.keys() if v not in processed_videos]
total = len(to_process)

with mp_holistic.Holistic(static_image_mode=False, model_complexity=0) as holistic:
    for i, video_name in enumerate(to_process, 1):
        video_path = ""
        for root, _, files in os.walk(EXTRACT_FOLDER):
            if video_name in files:
                video_path = os.path.join(root, video_name)
                break

        if not video_path: continue

        cap = cv2.VideoCapture(video_path)
        video_rows = [] # Bu videonun karelerini burada topla

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            frame = cv2.resize(frame, (320, 240))
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(frame_rgb)

            row = [video_name, label_dict[video_name], int(cap.get(cv2.CAP_PROP_POS_FRAMES))]

            def add_c(res, count):
                if res:
                    for lm in res.landmark: row.extend([lm.x, lm.y, lm.z])
                else: row.extend([0.0] * (count * 3))

            add_c(results.pose_landmarks, 33)
            add_c(results.right_hand_landmarks, 21)
            add_c(results.left_hand_landmarks, 21)
            video_rows.append(row)

        cap.release()

        # --- KRİTİK NOKTA: VİDEO BİTTİĞİ AN DOSYAYA YAZ ---
        if video_rows:
            pd.DataFrame(video_rows).to_csv(OUTPUT_FILE, mode='a', header=False, index=False)
            print(f"[{i}/{total}] Drive'a Kaydedildi: {video_name}")

Writing isleyici.py


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!python3.9 isleyici.py

--- RAR arşivi açılıyor... ---
patool: Extracting /content/drive/MyDrive/test.rar ...
patool: running /usr/bin/unrar x -- /content/drive/MyDrive/test.rar
patool:     with cwd='/content/test_videolari'
patool: ... /content/drive/MyDrive/test.rar extracted to `/content/test_videolari'.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[1/3742] Drive'a Kaydedildi: signer34_sample1_color.mp4
[2/3742] Drive'a Kaydedildi: signer34_sample2_color.mp4
[3/3742] Drive'a Kaydedildi: signer34_sample3_color.mp4
[4/3742] Drive'a Kaydedildi: signer34_sample4_color.mp4
[5/3742] Drive'a Kaydedildi: signer34_sample5_color.mp4
[6/3742] Drive'a Kaydedildi: signer34_sample6_color.mp4
[7/3742] Drive'a Kaydedildi: signer34_sample7_color.mp4
[8/3742] Drive'a Kaydedildi: signer34_sample8_color.mp4
[9/3742] Drive'a Kaydedildi: signer34_sample9_color.mp4
[10/3742] Drive'a Kaydedildi: signer34_sample10_color.mp4
[11/3742] Drive'a Kaydedildi: signer34_sample11_color.mp4
[12/3742] Drive'a Kaydedildi: signer34_